# **Laboratory of Wearable Biosignal Processing**

# **01 - Biosignals and Data Loading, Processing and Analysis**

## Before You Start

Open a terminal in the repository root folder and run:

```bash
git pull
```

This command updates your local copy of the course repository by downloading the latest changes from the remote repository, including this notebook and the data files used in the following exercises.

## Python Environment Setup (venv)

Before running the notebook code, open a terminal in the repository root and make sure the virtual environment `venv` is activated.

Windows PowerShell:

```powershell
.\venv\Scripts\Activate.ps1
```
macOS / Linux Terminal:

```terminal
source venv/bin/activate
```

Usually, when opening the terminal, this command is automatically executed after the first time you activated it.

After doing so, install some base packages with `pip`:

```bash
python -m pip install ast numpy pandas matplotlib scipy wfdb neurokit2
```

Recall that every time you need to add a new package that is used into your code, you must install it through the venv. Be sure that the `python/python3/python3.11` command is the same that you used for the venv configuration, otherwise `pip` will not work.

## Notebook Kernel Selection (First Time Only)

When you run notebook cells for the first time, select the Python kernel associated with your `venv` environment from the kernel dropdown menu (top-right of the notebook).

If you select a different kernel, packages installed in `venv` may not be available and imports can fail.

## Dataset: PTB-XL (Subset)

In this exercise, we use a subset of the **PTB-XL** dataset, containing data from the first 100 subjects.

PTB-XL is a large, publicly available clinical ECG dataset that includes 12-lead recordings and rich diagnostic metadata, making it suitable for signal processing and machine learning exercises.

Official dataset page: [PTB-XL](https://physionet.org/content/ptb-xl/1.0.3/)

## Metadata Files Overview

The file `ptbxl_database.csv` contains one row per ECG record. Some key attributes are:

- `ecg_id`: unique identifier of the ECG recording.
- `patient_id`: patient identifier (a patient may have multiple ECGs).
- `age`, `sex`, `height`, `weight`: basic demographic/anthropometric information.
- `recording_date`, `device`, `site`, `nurse`: acquisition context metadata.
- `report`: original clinical report text.
- `filename_lr`, `filename_hr`: relative paths to low-resolution (`records100`) and high-resolution (`records500`) waveform files.
- `scp_codes`: dictionary-like field with SCP diagnostic/rhythm statements and associated likelihood/weight.

The file `scp_statements.csv` is the lookup table for SCP codes:

- each row is an SCP code (for example `NORM`, `IMI`, `LAFB`).
- columns such as `description`, `diagnostic`, `form`, `rhythm`, `diagnostic_class`, and `diagnostic_subclass` provide clinical meaning and category.

How they are linked:

- read `scp_codes` from each row in `ptbxl_database.csv`.
- for every code appearing in `scp_codes`, match it to the corresponding index/code in `scp_statements.csv`.
- this enriches each ECG record with interpretable labels and class information.

## Data Extraction and First Visualization

In this section we:

1. Load and inspect PTB-XL metadata.
2. Load all ECG signals for a selected sampling rate (100 Hz or 500 Hz).
3. Select one reproducible random record (fixed seed).
4. Plot all 12 leads with 6 leads in the left column and 6 in the right column.

The helper function `get_record_filename(...)` is used to choose whether to load/plot the **100 Hz** or **500 Hz** version of the signal.

In [ ]:
# Imports for data loading, handling, plotting, filtering, and ECG peak analysis
import ast
import os
import wfdb

import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd

from scipy import signal

In [ ]:
# Root folder containing PTB-XL metadata and waveform files.
DATA_DIR = "../data"
# Fixed seed to keep random record selection reproducible across runs.
SEED = 42

# Output folder for all generated figures (grouped by notebook context).
NOTEBOOK_STEM = "01_biosignals_loading_processing_and_analysis"
NOTEBOOK_CONTEXT = os.path.basename(os.getcwd())

OUTPUTS_DIR = os.path.abspath(os.path.join("..", "outputs", f"{NOTEBOOK_CONTEXT}_{NOTEBOOK_STEM}"))
if not os.path.exists(OUTPUTS_DIR):
    os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Global counter to enforce ordered names: figure_01_..., figure_02_..., etc.
FIGURE_COUNTER = 1

In [ ]:
def save_figure(fig, suffix):
    """Save a matplotlib figure using the required sequential naming convention."""
    global FIGURE_COUNTER
    filename = f"figure_{FIGURE_COUNTER:02d}_{suffix}.png"
    output_path = os.path.join(OUTPUTS_DIR, filename)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    FIGURE_COUNTER += 1

def get_record_filename(row, sampling_rate):
    """
    ## Description
    Choose the PTB-XL record path according to the desired sampling rate.

    ## Parameters
    - `row`: a row of the PTB-XL metadata dataframe containing the record information.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the record.
    """

    # PTB-XL stores low-rate and high-rate paths in separate metadata columns
    # This function is an helper to be used to extract the correct path for loading a record at the desired sampling rate
    if sampling_rate == 100:
        return row["filename_lr"]
    if sampling_rate == 500:
        return row["filename_hr"]
    raise ValueError("sampling_rate must be 100 or 500")


def load_ecg_record(row, sampling_rate, data_dir):
    """
    ## Description
    Load one ECG record (WFDB) from PTB-XL metadata row.

    ## Parameters
    - `row`: a row of the PTB-XL metadata dataframe containing the record information.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the record.
    - `data_dir`: the root folder containing PTB-XL metadata and waveform files.
    """

    # Extract the path to the record file according to the desired sampling rate
    rel_path = get_record_filename(row, sampling_rate)

    # WFDB expects a path without extension (it resolves .hea/.dat files).
    full_path = f"{data_dir}/{rel_path}"

    return wfdb.rdrecord(full_path)  # Return the loaded record as a WFDB Record object


def load_all_ecg_records(metadata_df, sampling_rate, data_dir):
    """
    ## Description
    Load all ECG records at a given sampling rate and return a dict keyed by ecg_id.

    ## Parameters
    - `metadata_df`: the PTB-XL metadata dataframe containing information about all records.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the records.
    - `data_dir`: the root folder containing PTB-XL metadata and waveform files.
    """
    records_by_ecg_id = {}

    # Iterate metadata rows and cache each loaded record for quick random access.
    for _, row in metadata_df.iterrows():  # This helps in extracting the index and the row content at the same time
        # Take the ecg_id as the key for the loaded record
        ecg_id = int(row["ecg_id"])

        # Load the record using the helper function and store it in the dict as a wfdb Record object
        records_by_ecg_id[ecg_id] = load_ecg_record(row, sampling_rate, data_dir)

    return records_by_ecg_id


def plot_12_leads_two_columns(
    record,
    sampling_rate,
    patient_id,
    ecg_id,
    max_seconds=10.0,
    apply_standard_normalization=False,
):
    """
    ## Description
    Plot 12 ECG leads with 6 leads in the left column and 6 in the right column.

    ## Parameters
    - `record`: a WFDB Record object containing the ECG signal and metadata to plot.
    - `sampling_rate`: the sampling rate of the ECG signal (100 or 500 Hz for PTB-XL).
    - `patient_id`: the patient ID to display in the plot title.
    - `ecg_id`: the ECG ID to display in the plot title.
    - `max_seconds`: the maximum number of seconds to plot (must be <= 10.0).
    - `apply_standard_normalization`: whether to apply standard normalization (z-score) to each lead independently over the plotted window (default: False).
    """

    if max_seconds > 10.0:
        raise ValueError("max_seconds cannot be greater than 10.0")
    if max_seconds <= 0:
        raise ValueError("max_seconds must be greater than 0")

    # Paper-ready font sizes
    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 17

    # Extract the signal and lead names from the WFDB Record object
    signal = record.p_signal.copy()
    lead_names = record.sig_name

    # Restrict the visible window to improve readability on long recordings.
    n_samples = min(signal.shape[0], int(max_seconds * sampling_rate))
    t = np.arange(n_samples) / sampling_rate

    if apply_standard_normalization:
        # Normalize each lead independently over the plotted window.
        means = np.mean(signal[:n_samples, :], axis=0)
        stds = np.std(signal[:n_samples, :], axis=0)
        stds[stds == 0] = 1.0
        signal[:n_samples, :] = (signal[:n_samples, :] - means) / stds

    # Share y-axis only when normalized; otherwise keep independent scales.
    fig, axes = plt.subplots(
        6,
        2,
        figsize=(16, 16),
        sharex=True,
        sharey=apply_standard_normalization,
    )

    # Place leads 0..5 in the left column and 6..11 in the right column.
    for lead_idx in range(12):
        col = 0 if lead_idx < 6 else 1
        row = lead_idx if lead_idx < 6 else lead_idx - 6

        ax = axes[row, col]
        ax.plot(t, signal[:n_samples, lead_idx], linewidth=1.0)
        ax.set_title(lead_names[lead_idx], fontsize=title_fs)
        ax.set_ylabel("z-score" if apply_standard_normalization else "mV", fontsize=label_fs)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
        ax.grid(True, alpha=0.7)

    axes[5, 0].set_xlabel("Time [s]", fontsize=label_fs)
    axes[5, 1].set_xlabel("Time [s]", fontsize=label_fs)
    axes[0, 0].set_xlim(0, max_seconds)

    norm_label = "normalized" if apply_standard_normalization else "raw"
    fig.suptitle(
        f"PTB-XL ECG | patient_id={patient_id} | ecg_id={ecg_id} | fs={sampling_rate} Hz | {norm_label}",
        fontsize=suptitle_fs,
    )
    plt.tight_layout()
    save_figure(fig, "12_leads")
    plt.show()

In [ ]:
# Load the entire metadata table (one row per ECG exam).
metadata_path = f"{DATA_DIR}/ptbxl_database.csv"
ptbxl_df = pd.read_csv(metadata_path)

print(f"Metadata shape: {ptbxl_df.shape}")
print("First columns:", list(ptbxl_df.columns[:10]))
display(ptbxl_df.head(3))

In [ ]:
# Choose here: 100 for low-resolution or 500 for high-resolution.
sampling_rate_to_plot = 500

# Choose here: True to standardize each lead (z-score), False to keep raw amplitudes.
apply_standard_normalization = False

# Choose here: how many seconds of ECG to visualize in the final plot (max 10.0).
max_seconds_to_plot = 10.0

# Load all ECG signals once for the selected sampling rate.
all_records = load_all_ecg_records(
    ptbxl_df,
    sampling_rate=sampling_rate_to_plot,
    data_dir=DATA_DIR,
)

print(f"Loaded ECG records: {len(all_records)}")
print(f"Type of each object in the dictionary: {type(all_records[1])}") # Each value in the dict is a WFDB Record object containing the signal and metadata of one ECG exam

In [ ]:
# Use a deterministic RNG so every run selects the same example row.
rng = np.random.default_rng(SEED)
selected_idx = int(rng.integers(0, len(ptbxl_df)))
selected_row = ptbxl_df.iloc[selected_idx]

selected_patient_id = int(selected_row["patient_id"])
selected_ecg_id = int(selected_row["ecg_id"])
record = all_records[selected_ecg_id]

print(f"Selected row index (seed={SEED}): {selected_idx}")
print(f"Selected patient_id: {selected_patient_id}")
print(f"Selected ecg_id: {selected_ecg_id}")

**ECG Leads Scheme**

<div style="display: flex; justify-content: center; align-items: flex-start; gap: 16px; flex-wrap: wrap;">
  <img src="../images/Einthoven_Triangle.webp" alt="Einthoven Triangle" width="900">
  <img src="../images/Precordial_Leads.jpg" alt="Precordial Leads" width="507">
</div>

In [ ]:
# Plot up to the configured number of seconds of the selected 12-lead ECG.
plot_12_leads_two_columns(
    record,
    sampling_rate=sampling_rate_to_plot,
    patient_id=selected_patient_id,
    ecg_id=selected_ecg_id,
    max_seconds=max_seconds_to_plot,
    apply_standard_normalization=apply_standard_normalization,
)

## Step-by-Step ECG Filtering: Notch + Bandpass

In this section we apply two filters in sequence to the selected ECG record:

1. A **50 Hz notch filter** to attenuate power-line interference.
2. A **0.05-30 Hz bandpass filter** to retain the ECG-relevant frequency band.

Then we plot the same lead after each step to visually compare the effect of filtering.

In [ ]:
def apply_notch_filter(ecg_signal, fs, notch_freq=50.0, quality_factor=30.0):
    """
    ## Description
    Apply a notch filter (default 50 Hz) to attenuate power-line interference.
    
    ## Parameters
    - `ecg_signal`: the input ECG signal to filter (2D array: samples x leads).
    - `fs`: the sampling frequency of the ECG signal (e.g., 100 or 500 Hz for PTB-XL).
    - `notch_freq`: the frequency to notch out (default: 50.0 Hz, common power-line interference frequency).
    - `quality_factor`: the quality factor (Q) of the notch filter, controlling the bandwidth around the notch frequency (default: 30.0, higher means narrower notch).
    """
    # iirnotch returns numerator/denominator coefficients of the notch filter.
    b, a = signal.iirnotch(w0=notch_freq, Q=quality_factor, fs=fs)
    # filtfilt applies forward-backward filtering to avoid phase distortion.
    return signal.filtfilt(b, a, ecg_signal, axis=0)


def apply_bandpass_filter(ecg_signal, fs, lowcut=0.05, highcut=30.0, order=4):
    """
    ## Description
    Apply a Butterworth bandpass filter to retain ECG-relevant frequencies.
    
    ## Parameters
    - `ecg_signal`: the input ECG signal to filter (2D array: samples x leads).
    - `fs`: the sampling frequency of the ECG signal (e.g., 100 or 500 Hz for PTB-XL).
    - `lowcut`: the lower cutoff frequency of the bandpass filter (default: 0.05 Hz).
    - `highcut`: the upper cutoff frequency of the bandpass filter (default: 30.0 Hz).
    - `order`: the order of the Butterworth filter (default: 4).
    """
    nyquist = 0.5 * fs
    # Validate cutoff frequencies before filter design.
    if not (0 < lowcut < highcut < nyquist):
        raise ValueError("Bandpass bounds must satisfy 0 < lowcut < highcut < Nyquist")

    # scipy.signal.butter expects normalized cutoffs in [0, 1] relative to Nyquist.
    normalized_low = lowcut / nyquist
    normalized_high = highcut / nyquist
    b, a = signal.butter(order, [normalized_low, normalized_high], btype="bandpass")
    return signal.filtfilt(b, a, ecg_signal, axis=0)


def plot_filtering_steps(
    raw_signal,
    notch_filtered_signal,
    bandpass_filtered_signal,
    fs,
    lead_index=1,
    max_seconds=10.0,
    lead_name="Lead II",
    apply_standard_normalization=False,
):
    """
    ## Description
    Plot one lead across raw, notch-filtered, and bandpass-filtered stages.
    
    ## Parameters
    - `raw_signal`: the original ECG signal before filtering (2D array: samples x leads).
    - `notch_filtered_signal`: the ECG signal after applying the notch filter (same shape as raw_signal).
    - `bandpass_filtered_signal`: the ECG signal after applying the bandpass filter (same shape as raw_signal).
    - `fs`: the sampling frequency of the signals (e.g., 100 or 500 Hz for PTB-XL).
    - `lead_index`: the index of the lead to plot (default: 1 for Lead II).
    - `max_seconds`: the maximum number of seconds to plot (must be <= 10.0).
    """
    # Keep the same safety bound used in previous plotting sections.
    if max_seconds > 10.0:
        raise ValueError("max_seconds cannot be greater than 10.0")
    if max_seconds <= 0:
        raise ValueError("max_seconds must be greater than 0")

    # Paper-ready font sizes (same style as the 12-lead plotting function).
    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 16

    # Restrict all stages to the same visible time window.
    n_samples = min(raw_signal.shape[0], int(max_seconds * fs))
    t = np.arange(n_samples) / fs

    # Copy arrays to avoid in-place edits of original signals during optional normalization.
    stages = [
        ("Raw signal", raw_signal.copy()),
        ("After 50 Hz notch filter", notch_filtered_signal.copy()),
        ("After 0.05-30 Hz bandpass filter", bandpass_filtered_signal.copy()),
    ]

    if apply_standard_normalization:
        # Normalize each stage independently on the displayed window.
        normalized_stages = []
        for stage_name, stage_signal in stages:
            means = np.mean(stage_signal[:n_samples, :], axis=0)
            stds = np.std(stage_signal[:n_samples, :], axis=0)
            stds[stds == 0] = 1.0
            stage_signal[:n_samples, :] = (stage_signal[:n_samples, :] - means) / stds
            normalized_stages.append((stage_name, stage_signal))
        stages = normalized_stages

    # Share y-axis only when standardized, otherwise keep independent scales.
    fig, axes = plt.subplots(
        3,
        1,
        figsize=(16, 12),
        sharex=True,
        sharey=apply_standard_normalization,
    )

    # Plot the chosen lead across the three stages with consistent styling.
    for ax, (stage_name, stage_signal) in zip(axes, stages):
        ax.plot(t, stage_signal[:n_samples, lead_index], linewidth=1.1)
        ax.set_ylabel("z-score" if apply_standard_normalization else "Amplitude [mV]", fontsize=label_fs)
        ax.set_title(f"{stage_name} | {lead_name}", fontsize=title_fs)
        ax.grid(True, alpha=0.7)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)

    axes[-1].set_xlabel("Time [s]", fontsize=label_fs)
    axes[-1].set_xlim(0, max_seconds)

    norm_label = "normalized" if apply_standard_normalization else "raw"
    fig.suptitle(f"Filtering Stages Comparison | {norm_label}", fontsize=suptitle_fs)
    plt.tight_layout()
    save_figure(fig, "filtering_steps")
    plt.show()

In [ ]:
# Use the selected record already loaded in previous cells.
raw_signal = record.p_signal.copy()
fs = sampling_rate_to_plot

# Step 1: remove narrow-band power-line interference at 50 Hz.
# The function expects a 2D array (samples x leads), so we pass the full 12-lead matrix.
signal_after_notch = apply_notch_filter(
    raw_signal,
    fs=fs,
    notch_freq=50.0,
    quality_factor=30.0,
)

# Step 2: keep the ECG band that contains most morphology information.
# 0.05-30 Hz is a common range for baseline/signal denoising in many examples.
signal_after_bandpass = apply_bandpass_filter(
    signal_after_notch,
    fs=fs,
    lowcut=0.05,
    highcut=30.0,
    order=4,
)

ecg_leads = {
    'I': 0,
    'II': 1,
    'III': 2,
    'aVR': 3,
    'aVL': 4,
    'aVF': 5,
    'V1': 6,
    'V2': 7,
    'V3': 8,
    'V4': 9,
    'V5': 10,
    'V6': 11,
}

# Choose one derivation to visualize intermediate stages
lead_index_to_plot = ecg_leads['I']  # Lead II is commonly used for rhythm analysis and has good SNR
lead_name_to_plot = record.sig_name[lead_index_to_plot]

# Compare raw vs notch-filtered vs bandpass-filtered signal on the same time window.
plot_filtering_steps(
    raw_signal,
    signal_after_notch,
    signal_after_bandpass,
    fs=fs,
    lead_index=lead_index_to_plot,
    max_seconds=max_seconds_to_plot,
    lead_name=lead_name_to_plot,
    apply_standard_normalization=apply_standard_normalization,
)

## Frequency-Domain Check: FFT Across Filtering Stages

To better understand what each filter is doing, inspect the same lead in the frequency domain.

Recommended interpretation:

1. **Raw signal**: may show broad noise and a visible component around 50 Hz.
2. **After notch**: the component around 50 Hz should be attenuated.
3. **After bandpass (0.05-30 Hz)**: most energy above 30 Hz should be strongly reduced.

Guide lines at **30 Hz** and **50 Hz** are added to help interpret the filter effects.

In [ ]:
def compute_single_sided_fft(signal_1d, fs):
    """
    ## Description
    Return one-sided FFT frequencies and amplitude spectrum for a 1D signal.

    ## Parameters
    - `signal_1d`: the input signal to analyze (1D array).
    - `fs`: the sampling frequency of the signal (e.g., 100 or 500 Hz for PTB-XL).
    """
    n = len(signal_1d)

    # DC removal: subtract the mean before spectral analysis.
    detrended = signal_1d - np.mean(signal_1d)
    window = np.hanning(n)
    windowed = detrended * window

    fft_vals = np.fft.rfft(windowed)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)

    # Amplitude scaling for one-sided spectrum (window-corrected).
    scale = 2.0 / np.sum(window)
    amplitude = np.abs(fft_vals) * scale
    return freqs, amplitude


def plot_fft_filtering_stages(
    raw_signal,
    notch_filtered_signal,
    bandpass_filtered_signal,
    fs,
    lead_index=1,
    max_freq_to_show=80.0,
    lead_name="Lead II",
):
    """
    ## Description
    Plot FFT magnitude of one lead across raw, notch-filtered, and bandpass-filtered stages.

    ## Parameters
    - `raw_signal`: the original ECG signal before filtering (2D array: samples x leads).
    - `notch_filtered_signal`: the ECG signal after applying the notch filter (same shape as raw_signal).
    - `bandpass_filtered_signal`: the ECG signal after applying the bandpass filter (same shape as raw_signal).
    - `fs`: the sampling frequency of the signals (e.g., 100 or 500 Hz for PTB-XL).
    - `lead_index`: the index of the lead to analyze (default: 1 for Lead II).
    - `max_freq_to_show`: the maximum frequency to display on the x-axis (default: 80.0 Hz).
    - `lead_name`: the name of the lead to display in the plot titles (default: "Lead II").
    """
    stages = [
        ("Raw signal", raw_signal),
        ("After 50 Hz notch filter", notch_filtered_signal),
        ("After 0.05-30 Hz bandpass filter", bandpass_filtered_signal),
    ]

    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 16

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True, sharey=False)

    for ax, (stage_name, stage_signal) in zip(axes, stages):
        signal_1d = stage_signal[:, lead_index]
        mean_before_detrend = float(np.mean(signal_1d))
        freqs, amp = compute_single_sided_fft(signal_1d, fs)
        ax.plot(freqs, amp, linewidth=1.1)

        ax.set_title(f"{stage_name} | {lead_name}", fontsize=title_fs)
        ax.set_ylabel("Amplitude", fontsize=label_fs)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
        ax.grid(True, alpha=0.7)
        ax.set_xlim(0.0, min(max_freq_to_show, fs / 2.0))

        # Visual guides for the bandpass upper cutoff and powerline frequency.
        ax.axvline(30.0, color="#34495e", linestyle="--", linewidth=1.2, alpha=0.9)
        ax.axvline(50.0, color="#8e44ad", linestyle="--", linewidth=1.2, alpha=0.9)

        # Explicit reminder that DC was removed before computing FFT.
        ax.text(
            0.99,
            0.93,
            f"DC removed (mean={mean_before_detrend:.3e})",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=10,
            bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.9},
        )

    axes[-1].set_xlabel("Frequency [Hz]", fontsize=label_fs)
    axes[-1].set_xlim(-0.5, max_freq_to_show + 0.5)
    fig.suptitle("FFT Comparison Across Filtering Stages", fontsize=suptitle_fs)
    plt.tight_layout()
    save_figure(fig, "fft_filtering_stages")
    plt.show()

In [ ]:
# Reuse the already selected lead from the filtering stage visualization.
# If not available, default to Lead II (index 1).
try:
    lead_idx_fft = lead_index_to_plot
except NameError:
    lead_idx_fft = 1

lead_name_fft = record.sig_name[lead_idx_fft]

plot_fft_filtering_stages(
    raw_signal,
    signal_after_notch,
    signal_after_bandpass,
    fs=fs,
    lead_index=lead_idx_fft,
    max_freq_to_show=55.0,
    lead_name=lead_name_fft,
)

## R-Peak Extraction With NeuroKit2

In this part, your goal is to detect R-peaks on **all 12 derivations** in a robust and reproducible way. To do this, you can use the NeuroKit2 package (link to the documentation: [NeuroKit2](https://neuropsychology.github.io/NeuroKit/)).

Retrieve the needed function to understand which and how they return outputs that could be useful for you.

Suggested workflow:

1. Start from a reasonably denoised signal (for example after notch + bandpass filtering).
2. For each lead, clean the 1D ECG with `nk.ecg_clean(...)`.
3. Detect R-peaks with `nk.ecg_peaks(...)` lead-by-lead.
4. Plot all derivations in one figure and overlay detected R-peaks for visual validation.
5. Compute lead-wise RR summaries and check whether values are physiologically plausible.

***Tip**: always validate detected peaks visually before using RR/HR metrics for further analysis.*

## Bonus: Subject-Level Mean RR and Healthy vs Non-Healthy Comparison

In this bonus section, you will estimate the **mean RR interval for each subject** and compare distributions between healthy and non-healthy groups.

Grouping criterion used in this exercise (strict diagnostic rule):

- A record is considered **healthy** only if its diagnostic SCP set is exactly `{NORM}`.
- Otherwise, it is considered **non-healthy**.
- A subject is considered **healthy** only if all analyzed records for that subject are healthy.

Then we compute one mean RR value per subject and show a boxplot for the two groups.